In [1]:
import open3d as o3d
import numpy as np
import json
import os
from pathlib import Path
from glob import glob
from PIL import Image
import cv2
import networkx as nx
import matplotlib.pyplot as plt
import random
from collections import defaultdict

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# 1. Load the full mesh

In [2]:
# scene_dir = 'ScanNet_Data/data/56a0ec536c'

# Change the scene: c9abde4c4b - toilet
# scene: 285efbc7cf - kitchen
scene_dir = 'ScanNet_Data/data/0a7cc12c0e'

# Change the scene: 0a7cc12c0e - bedroom
# scene_dir = 'ScanNet_Data/data/0a7cc12c0e'

mesh_file = scene_dir + '/scans/mesh_aligned_0.05_semantic.ply'
segments_file = scene_dir + '/scans/segments.json'
anno_file = scene_dir + '/scans/segments_anno.json'
output_dir = scene_dir + '/extracted_individual_objects'
os.makedirs(output_dir, exist_ok=True)
colmap_dir = os.path.join(scene_dir, 'dslr/colmap')
image_dir = os.path.join(scene_dir, 'dslr/resized_images')
region_crop_dir = scene_dir + '/region_crops'
os.makedirs(region_crop_dir, exist_ok=True)
region_dir = scene_dir + '/regions'
os.makedirs(region_dir, exist_ok=True)
region_top5_dir = os.path.join(scene_dir, 'region_cropped_top5')
os.makedirs(region_top5_dir, exist_ok=True)

In [3]:
mesh = o3d.io.read_triangle_mesh(mesh_file)
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
segment_ids = np.asarray(json.load(open(segments_file))['segIndices'])
anno_data = np.asarray(json.load(open(anno_file))['segGroups'])

# 2. Extract Individual Objects

In [4]:
segment_to_object = {}
object_to_label = {}
for object in anno_data:
    object_id = object['objectId']
    label = object['label']
    segments = object['segments']
    for segment_id in segments:
        segment_to_object[segment_id] = object_id
    object_to_label[object_id] = label

In [5]:
unique_instances = set(segment_to_object.values())
print(f'There are {len(unique_instances)} unique instances.')

There are 140 unique instances.


# 3. Centroid, Bounding box, PCA, etc.

In [ ]:
import numpy as np
import json
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from collections import defaultdict
import open3d as o3d

# === 加载文件 ===
mesh = o3d.io.read_triangle_mesh(mesh_file)
mesh.compute_vertex_normals()
vertices = np.asarray(mesh.vertices)
normals = np.asarray(mesh.vertex_normals)
segment_ids = np.asarray(json.load(open(segments_file))['segIndices'])
anno_data = np.asarray(json.load(open(anno_file))['segGroups'])

# === 加速结构：segment_id → vertex indices ===
segment_to_vertex_indices = defaultdict(list)
for i, seg_id in enumerate(segment_ids):
    segment_to_vertex_indices[seg_id].append(i)

# === 构建 object 映射 ===
segment_to_object = {}
object_to_label = {}
for obj in anno_data:
    obj_id = obj['objectId']
    label = obj['label']
    segments = obj['segments']
    for seg_id in segments:
        segment_to_object[seg_id] = obj_id
    object_to_label[obj_id] = label

unique_instances = set(segment_to_object.values())

# === 特征提取（快速版）===
instance_features = []
instance_ids = []
instance_labels = []

for object_id in unique_instances:
    # 聚合所有 vertex indices for this object
    segment_ids_in_object = [sid for sid, oid in segment_to_object.items() if oid == object_id]
    vertex_indices = []
    for sid in segment_ids_in_object:
        vertex_indices.extend(segment_to_vertex_indices.get(sid, []))
    
    if len(vertex_indices) < 10:
        continue  # 忽略太小的 instance

    verts = vertices[vertex_indices]
    norms = normals[vertex_indices]

    # === 特征提取（快速+稳定）===
    centroid = verts.mean(axis=0)
    bbox_size = verts.max(axis=0) - verts.min(axis=0)

    # 替代 PCA：最大边长表示 elongation
    elongation = np.linalg.norm(bbox_size)

    # Normal 均值代替 histogram
    normal_mean = norms.mean(axis=0)

    # 最终特征
    feat = np.concatenate([centroid, bbox_size, [elongation], normal_mean])
    
    instance_ids.append(object_id)
    instance_labels.append(object_to_label[object_id])
    instance_features.append(feat)


# 4. Clustering

In [14]:
X = np.array(instance_features)
X_scaled = StandardScaler().fit_transform(X)

clustering = AgglomerativeClustering(n_clusters=4)
cluster_labels = clustering.fit_predict(X_scaled)

# objectId → clusterId 映射
object_to_cluster = {oid: cid for oid, cid in zip(instance_ids, cluster_labels)}


In [15]:
num_clusters = len(set(object_to_cluster.values()))
print(f"Number of clusters: {num_clusters}")

Number of clusters: 4


In [16]:
object_to_centroid = {}
for i, file in enumerate(os.listdir(output_dir)):
    mesh = o3d.io.read_triangle_mesh(os.path.join(output_dir, file))
    vertices = mesh.vertices
    triangles = mesh.triangles
    centroid = np.mean(vertices, axis=0)
    object_to_centroid[i] = centroid

In [17]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np

if object_to_cluster:
    min_cluster_id = min(object_to_cluster.values())
    max_cluster_id = max(object_to_cluster.values())
    num_distinct_clusters = max_cluster_id - min_cluster_id + 1
else:
    min_cluster_id = 0
    num_distinct_clusters = 1

colors_cmap = plt.get_cmap('tab20', num_distinct_clusters)

x, y, z, color_list = [], [], [], []

for obj_id, centroid in object_to_centroid.items():
    x.append(centroid[0])
    y.append(centroid[1])
    z.append(centroid[2])

    cluster_id = object_to_cluster.get(obj_id, min_cluster_id)
    color_index = cluster_id - min_cluster_id
    rgba_color = colors_cmap(color_index)
    color_list.append(f"rgba({int(rgba_color[0]*255)}, {int(rgba_color[1]*255)}, {int(rgba_color[2]*255)}, {rgba_color[3]})")

fig = go.Figure(data=[go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=4, color=color_list),
    text=[f'Object {obj_id}. Cluster {object_to_cluster.get(obj_id, "N/A")}' for obj_id in object_to_centroid.keys()]
)])

fig.update_layout(
    title="3D Object Clusters (Interactive)",
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z')
)
fig.show()


Codes below not used any more

In [30]:
from sklearn.cluster import DBSCAN
object_ids = list(object_to_centroid.keys())
centroids = np.array([object_to_centroid[obj_id] for obj_id in object_to_centroid])

In [31]:
# TODO: adjust the params of dbscan
dbscan = DBSCAN(eps=0.8, min_samples=4)
regions = dbscan.fit_predict(centroids)
object_to_regions = {obj_id : region_id for obj_id, region_id in zip(object_ids, regions)}

In [32]:
# Try HDBScan
'''
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
regions = clusterer.fit_predict(centroids)
object_to_regions = {obj_id: region_id for obj_id, region_id in zip(object_ids, regions)}
'''

'\nimport hdbscan\n\nclusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)\nregions = clusterer.fit_predict(centroids)\nobject_to_regions = {obj_id: region_id for obj_id, region_id in zip(object_ids, regions)}\n'

In [34]:
# Change: check the clustering
import plotly.graph_objects as go
import numpy as np

if object_to_regions:
    min_cluster_id = min(object_to_regions.values())
    max_cluster_id = max(object_to_regions.values())
    num_distinct_clusters = max_cluster_id - min_cluster_id + 1
else:
    min_cluster_id = 0
    num_distinct_clusters = 1 # Default to 1 if no regions are found

colors_cmap = plt.get_cmap('tab20', num_distinct_clusters)

x, y, z, color_list = [], [], [], []

for obj_id, centroid in object_to_centroid.items():
    x.append(centroid[0])
    y.append(centroid[1])
    z.append(centroid[2])

    cluster_id = object_to_regions.get(obj_id, min_cluster_id) # Use .get() with a default for robustness
    
    # Map the cluster_id to a 0-based index for the colormap
    # For example, if min_cluster_id is -1, then -1 becomes index 0, 0 becomes index 1, etc.
    color_index = cluster_id - min_cluster_id
    
    # Get the RGBA color from the colormap
    rgba_color = colors_cmap(color_index)
    
    # Format the color as a string suitable for Plotly's marker color
    # Plotly expects 'rgba(r,g,b,a)'
    color_list.append(f"rgba({int(rgba_color[0]*255)}, {int(rgba_color[1]*255)}, {int(rgba_color[2]*255)}, {rgba_color[3]})")

fig = go.Figure(data=[go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=4, color=color_list), # Use the correctly generated color_list
    text=[f'Object {obj_id}. Region {object_to_regions.get(obj_id, "N/A")}' for obj_id in object_to_centroid.keys()]
)])

fig.update_layout(
    title="3D Object Clusters (Interactive)",
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z')
)
fig.show()

In [35]:
from collections import Counter

value_counts = Counter(object_to_regions.values())
print(value_counts)

Counter({1: 39, 2: 31, 0: 24, 3: 20, 4: 14, -1: 12})


# 3. Combine the individual objects of each region together into submesh

In [ ]:
cluster_to_mesh = defaultdict(o3d.geometry.TriangleMesh)

for obj_file in os.listdir(output_dir):
    if not obj_file.endswith('.ply'):
        continue
    obj_id = int(os.path.splitext(obj_file)[0].split("_")[1])
    cluster_id = object_to_regions[obj_id-1]
    
    mesh = o3d.io.read_triangle_mesh(os.path.join(output_dir, obj_file))
    cluster_to_mesh[cluster_id] += mesh  

In [18]:
for cluster_id, mesh in cluster_to_mesh.items():
    o3d.io.write_triangle_mesh(os.path.join(region_dir, f"merged_cluster_{cluster_id}.ply"), mesh)

# 4. Map the Region Mesh into 2D images

In [21]:
def read_cameras_text(path):
    cameras = {}
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            elems = line.strip().split()
            camera_id, model, width, height, fx, fy, cx, cy = elems[:8]
            cameras[int(camera_id)] = {
                'fx': float(fx), 'fy': float(fy), 'cx': float(cx), 'cy': float(cy),
                'width': int(width), 'height': int(height)
            }
    return cameras

def read_images_text(path):
    images = {}
    with open(path, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith("#") or line == "":
            i += 1
            continue
        elems = line.split()
        if len(elems) < 10:
            i += 1
            continue  # Skip invalid lines
        try:
            image_id = int(elems[0])
            qvec = np.array(list(map(float, elems[1:5])))
            tvec = np.array(list(map(float, elems[5:8])))
            camera_id = int(elems[8])
            image_name = elems[9]
            images[image_name] = {'qvec': qvec, 'tvec': tvec, 'camera_id': camera_id}
            i += 2  # Skip the following line (2D points) in COLMAP text format
        except ValueError:
            i += 1
            continue
    return images

def qvec2rotmat(qvec):
    w, x, y, z = qvec
    return np.array([
        [1-2*y**2-2*z**2, 2*x*y-2*z*w, 2*x*z+2*y*w],
        [2*x*y+2*z*w, 1-2*x**2-2*z**2, 2*y*z-2*x*w],
        [2*x*z-2*y*w, 2*y*z+2*x*w, 1-2*x**2-2*y**2]
    ])

In [53]:
cameras = read_cameras_text(os.path.join(colmap_dir, 'cameras.txt'))
images = read_images_text(os.path.join(colmap_dir, 'images.txt'))
region_scene_dir = os.path.join(scene_dir, 'region_scene')
os.makedirs(region_scene_dir, exist_ok=True)
region_scene_top5_dir = os.path.join(scene_dir, 'region_scene_top5')
os.makedirs(region_scene_top5_dir, exist_ok=True)

# Save the projection area of all images for each obj
object_areas = defaultdict(list)

# Loop through all extracted regions
for obj_file in os.listdir(region_dir):
    if not obj_file.endswith('.ply'):
        continue
    obj_path = os.path.join(region_dir, obj_file)
    obj_mesh = o3d.io.read_triangle_mesh(obj_path)
    vertices = np.asarray(obj_mesh.vertices)
    print(f"Processing object: {obj_file}")

    for img_name, img_info in images.items():
        img_path = os.path.join(image_dir, img_name)
        if not os.path.exists(img_path):
            continue

        qvec, tvec, cam_id = img_info['qvec'], img_info['tvec'], img_info['camera_id']
        cam = cameras[cam_id]
        R = qvec2rotmat(qvec)
        t = np.array(tvec).reshape((3,1))
        K = np.array([[cam['fx'], 0, cam['cx']],
                      [0, cam['fy'], cam['cy']],
                      [0, 0, 1]])
        w, h = cam['width'], cam['height']

        # Make the projection from 3D to 2D
        verts_cam = (R @ vertices.T).T + t.T
        proj = (K @ verts_cam.T).T
        u, v, z = proj[:, 0] / proj[:, 2], proj[:, 1] / proj[:, 2], proj[:, 2]
        valid = (z > 0) & (u >= 0) & (u < w) & (v >= 0) & (v < h)
        if np.sum(valid) == 0:
            continue

        min_u, max_u = int(np.min(u[valid])), int(np.max(u[valid]))
        min_v, max_v = int(np.min(v[valid])), int(np.max(v[valid]))

        # padding
        padding = 0
        min_u = max(0, min_u - padding)
        max_u = min(w, max_u + padding)
        min_v = max(0, min_v - padding)
        max_v = min(h, max_v + padding)
        if max_u > min_u and max_v > min_v:
            area = (max_u - min_u) * (max_v - min_v)
            object_areas[obj_file].append((area, img_name, min_u, max_u, min_v, max_v))

            # Save all images
            image = np.array(Image.open(img_path))
            name = f"{os.path.splitext(obj_file)[0]}_{os.path.splitext(img_name)[0]}.jpg"
            path = os.path.join(region_scene_dir, name)
            cv2.imwrite(path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        else:
            print(f"Skipping invalid crop for {img_name}")

Processing object: merged_cluster_2.ply
Processing object: merged_cluster_3.ply
Processing object: merged_cluster_4.ply
Processing object: merged_cluster_0.ply
Processing object: merged_cluster_-1.ply
Processing object: merged_cluster_1.ply


In [54]:
region_scene = defaultdict(list)
for obj_file, views in object_areas.items():
    # extract Top-5 projections
    views_sorted = sorted(views, key=lambda x: x[0], reverse=True)
    top_5_view = views_sorted[:5]
    for area, img_name, min_u, max_u, min_v, max_v in top_5_view:
        img_path = os.path.join(image_dir, img_name)
        image = np.array(Image.open(img_path))
        name = f"{os.path.splitext(obj_file)[0]}_{os.path.splitext(img_name)[0]}.jpg"
        path = os.path.join(region_scene_top5_dir, name)
        cv2.imwrite(path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        print(f"Saved top5 views: {name}")
        region_scene[obj_file].append((path, min_u, max_u, min_v, max_v))

Saved top5 views: merged_cluster_2_DSC06191.jpg
Saved top5 views: merged_cluster_2_DSC06184.jpg
Saved top5 views: merged_cluster_2_DSC06183.jpg
Saved top5 views: merged_cluster_2_DSC06182.jpg
Saved top5 views: merged_cluster_2_DSC06180.jpg
Saved top5 views: merged_cluster_3_DSC06184.jpg
Saved top5 views: merged_cluster_3_DSC05889.jpg
Saved top5 views: merged_cluster_3_DSC05884.jpg
Saved top5 views: merged_cluster_3_DSC05883.jpg
Saved top5 views: merged_cluster_3_DSC05882.jpg
Saved top5 views: merged_cluster_4_DSC06127.jpg
Saved top5 views: merged_cluster_4_DSC06126.jpg
Saved top5 views: merged_cluster_4_DSC06116.jpg
Saved top5 views: merged_cluster_4_DSC06115.jpg
Saved top5 views: merged_cluster_4_DSC06094.jpg
Saved top5 views: merged_cluster_0_DSC06162.jpg
Saved top5 views: merged_cluster_0_DSC06161.jpg
Saved top5 views: merged_cluster_0_DSC06160.jpg
Saved top5 views: merged_cluster_0_DSC06158.jpg
Saved top5 views: merged_cluster_0_DSC06157.jpg
Saved top5 views: merged_cluster_-1_DSC0

# 5. Generate Caption for the region

In [14]:
from lmdeploy import pipeline, TurbomindEngineConfig, ChatTemplateConfig
from lmdeploy.vl.constants import IMAGE_TOKEN

model = 'OpenGVLab/InternVL3-2B'
pipe = pipeline(model, backend_config=TurbomindEngineConfig(session_len=16384, tp=1), chat_template_config=ChatTemplateConfig(model_name='internvl2_5'))

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

2025-06-10 16:09:10,306 - lmdeploy - WARNING - turbomind.py:252 - get 255 model params


[TM][WARNING] [LlamaTritonModel] `max_context_token_num` is not set, default to 16384.


2025-06-10 16:09:11,311 - lmdeploy - WARNING - tokenizer.py:499 - The token <|action_end|>, its length of indexes [27, 91, 1311, 6213, 91, 29] is over than 1. Currently, it can not be used as stop words


In [ ]:
# Numbering images improves multi-image conversations
question = f'Image: {IMAGE_TOKEN}\n'
question += '''
You are provided with a view of a specific region within a 3D scene. 
This region is from ({min_u},{min_v}) to ({max_u},{max_v}).
Describe the spatial location of the region within the entire scene and the relationship between this region and the scene,
focusing on spatial arrangement, functional interaction, and contextual relevance, not the detailed object.
'''
print(question)

Image: <IMAGE_TOKEN>

You are provided with a view of a specific region within a 3D scene. 
This region is from ({min_u},{min_v}) to ({max_u},{max_v}).
Describe the spatial location of the region within the entire scene and the relationship between this region and the scene,
focusing on spatial arrangement, functional interaction, and contextual relevance.



In [16]:
folder_path = f'{scene_dir}/region_scene_top5'
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

In [19]:
region_images = []
num_regions = np.max(list(object_to_regions.values()))
for i in range(num_regions+2):
    folder_path = f'{scene_dir}/region_scene_top5'
    image_path = [f'{scene_dir}/region_scene_top5/{path}' for path in file_list if f'cluster_{i-1}_' in path]
    filename = os.path.basename(image_path[0])
    parts = filename.split('_')
    region = parts[2]
    region_images.append(
        {
            f'Region_{region}': image_path[:1]
        }
    )
print(region_images)

[{'Region_-1': ['ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_-1_DSC05899.jpg']}, {'Region_0': ['ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_0_DSC06157.jpg']}, {'Region_1': ['ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_1_DSC06127.jpg']}, {'Region_2': ['ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_2_DSC06183.jpg']}, {'Region_3': ['ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_3_DSC05882.jpg']}, {'Region_4': ['ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_4_DSC06127.jpg']}]


In [39]:
print(region_scene.get('merged_cluster_2.ply')[0])

('ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_2_DSC06191.jpg', 0, 1751, 0, 1167)


In [51]:
print(region_scene.get('merged_cluster_2.ply'))

[('ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_2_DSC06191.jpg', 0, 1751, 0, 1167), ('ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_2_DSC06184.jpg', 0, 1751, 0, 1167), ('ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_2_DSC06183.jpg', 0, 1751, 0, 1167), ('ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_2_DSC06182.jpg', 0, 1751, 0, 1167), ('ScanNet_Data/data/0a7cc12c0e/region_scene_top5/merged_cluster_2_DSC06180.jpg', 0, 1751, 0, 1167)]


In [70]:
region_captions = defaultdict(list)
for i in range(max(object_to_regions.values())+2):
    images_tuples = region_scene.get(f'merged_cluster_{i-1}.ply')
    for t in images_tuples:
        image_path, min_u, min_v, max_u, max_v = t
        image = Image.open(image_path)
        question = f'Image: {IMAGE_TOKEN}\n'
        question += '''
        You are provided with an image showing a specific region within a 3D scene, defined by the bounding box coordinates ({min_u},{min_v}) to ({max_u},{max_v}).

        Describe this region's spatial location in the entire scene and its primary function or purpose. Focus on the **type of space** it represents, the **activities it facilitates**, and its **overall arrangement** relative to other parts of the scene.

        Use **definitive and factual language**. **Do NOT use words that express uncertainty** such as "likely," "possibly," "appears to be," "suggests," "indicates," or "implies."

        Crucially, **do not mention any specific objects or their details**.

        The output format should be:
        Region: Region Caption
        Region-Scene Relationship.
        '''
        question = question.format(min_u = min_u, min_v = min_v, max_u = max_u, max_v = max_v)
        response = pipe((question, image))
        region_captions[f'merged_cluster_{i-1}.ply'].append(response.text)

In [71]:
region_captions

defaultdict(list,
            {'merged_cluster_-1.ply': ['Region: The desk area.\nRegion-Scene Relationship: This region represents a workspace within the room. It is positioned against the wall where the calendar is displayed, facilitating activities such as studying, working, or organizing tasks. The desk provides a surface for placing items like a fan, a water bottle, a smartphone, and a magazine, indicating a setting where someone might engage in tasks that require a bit of comfort and convenience. The arrangement suggests a functional space designed for productivity or casual work.',
              'Region: The desk area.\nRegion-Scene Relationship: This region represents a workspace within a room. It is located on the left side of the image and is adjacent to a staircase leading upwards. The desk facilitates activities such as studying, working, or organizing tasks. The desk is positioned against a wall with a calendar and a fan, indicating a functional and organized space for pro

In [80]:
output_filename = 'region_captions.json'
try:
    with open(output_filename, 'w', encoding='utf-8') as f:
        # Use json.dump to write the dictionary to the file
        # indent=4 makes the JSON human-readable with indentation
        # ensure_ascii=False handles non-ASCII characters (like emojis, some foreign text) correctly
        json.dump(region_captions, f, indent=4, ensure_ascii=False)
    print(f"Successfully saved region captions to {output_filename}")
except Exception as e:
    print(f"Error saving region captions: {e}")


Successfully saved region captions to region_captions.json


In [ ]:
try:
    with open(output_filename, 'r', encoding='utf-8') as f:
        # Load the data into the new variable name 'region_level_captions' as requested.
        region_level_captions = json.load(f)
    print(f"\n--- Successfully Loaded Data from {output_filename} ---")
except FileNotFoundError:
    print(f"Error: The file '{output_filename}' was not found. Make sure it was saved correctly.")
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from '{output_filename}'. The file might be empty or corrupted.")
except Exception as e:
    print(f"An unexpected error occurred during loading: {e}")



--- Successfully Loaded Data from region_captions.json ---
{'merged_cluster_-1.ply': ['Region: The desk area.\nRegion-Scene Relationship: This region represents a workspace within the room. It is positioned against the wall where the calendar is displayed, facilitating activities such as studying, working, or organizing tasks. The desk provides a surface for placing items like a fan, a water bottle, a smartphone, and a magazine, indicating a setting where someone might engage in tasks that require a bit of comfort and convenience. The arrangement suggests a functional space designed for productivity or casual work.', 'Region: The desk area.\nRegion-Scene Relationship: This region represents a workspace within a room. It is located on the left side of the image and is adjacent to a staircase leading upwards. The desk facilitates activities such as studying, working, or organizing tasks. The desk is positioned against a wall with a calendar and a fan, indicating a functional and organiz

In [38]:
!nvidia-smi

Tue Jun 10 16:08:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.03              Driver Version: 575.51.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080        Off |   00000000:01:00.0 Off |                  N/A |
| 30%   56C    P8             17W /  320W |    9829MiB /  10240MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----